# 🩺 Diabetes Prediction & Clinical Risk Stratification Analysis

## 1. Problem Overview & Clinical Motivation
Diabetes mellitus is a chronic metabolic disorder characterized by high blood glucose levels over a prolonged period. Early diagnosis and proactive lifestyle interventions can dramatically reduce the risk of long-term complications including cardiovascular disease, neuropathy, nephropathy, and retinopathy.

### **Objectives:**
1. **Exploratory Data Analysis (EDA)**: Inspect distributions, missing values (physiological zeros), and correlations.
2. **Data Preparation**: Address physiological zeros with Iterative (MICE) & KNN Imputation, outlier handling, feature scaling, and synthetic class balancing (SMOTE).
3. **Feature Engineering**: Derive metabolic risk scores, HOMA-IR proxy, and Age-Glucose risk indices.
4. **Multi-Model Benchmark**: Train and evaluate Logistic Regression, SVM, Random Forest, Gradient Boosting, XGBoost, LightGBM, and Stacking/Voting Ensembles via 10-Fold Stratified Cross-Validation.
5. **Model Evaluation & Interpretation**: Analyze ROC-AUC, Sensitivity/Recall, Precision, Confusion Matrices, and Feature Importance.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visual style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Dataset Loading & Initial Exploration

In [ ]:
import sys
sys.path.append('..')
from src.data_loader import load_or_download_dataset

df = load_or_download_dataset(data_dir='../data', filename='diabetes.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

In [ ]:
df.info()
df.describe().T

### Target Variable Distribution (Class Balance)

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='Outcome', palette=['#3498db', '#e74c3c'])
plt.title('Outcome Class Distribution (0 = Non-Diabetic, 1 = Diabetic)', fontweight='bold')
plt.xlabel('Diagnosis Outcome')
plt.ylabel('Patient Count')
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')
plt.show()

## 3. Data Cleaning: Handling Physiological Zeros
In the Pima dataset, values of 0 for variables like **Glucose, BloodPressure, SkinThickness, Insulin, and BMI** are physiologically impossible and represent missing data.

In [ ]:
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
zero_counts = {col: (df[col] == 0).sum() for col in zero_cols}
zero_pct = {col: f'{(df[col] == 0).mean() * 100:.1f}%' for col in zero_cols}

missing_summary = pd.DataFrame({'Zero Count': zero_counts, 'Percentage Missing': zero_pct})
print('Missing / Physiological Zero Summary:')
missing_summary

## 4. Feature Correlations & Pairwise Analysis

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix', fontweight='bold')
plt.show()

## 5. End-to-End Data Preparation & Multi-Model Pipeline Training

In [ ]:
from src.preprocessor import DataPreparationPipeline
from src.model_trainer import ModelTrainer
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Outcome'])
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

pipeline = DataPreparationPipeline(imputer_strategy='iterative', scaler_strategy='robust', resampler_strategy='smote')
X_train_proc, y_train_res, feature_names = pipeline.fit_transform(X_train, y_train)
X_test_proc = pipeline.transform(X_test)

trainer = ModelTrainer(random_state=42, models_dir='../models')
cv_results = trainer.evaluate_cv(X_train_proc, y_train_res, n_splits=10)
cv_results

## 6. Best Model Evaluation & Diagnostic Visualizations

In [ ]:
best_name, best_model = trainer.train_and_select_best(X_train_proc, y_train_res)
print(f'Optimal Performing Model: {best_name}')

In [ ]:
from src.evaluate import evaluate_predictions, plot_evaluation_suite

test_metrics = []
for name, model in trainer.models.items():
    y_pred = model.predict(X_test_proc)
    y_prob = model.predict_proba(X_test_proc)[:, 1] if hasattr(model, 'predict_proba') else None
    m = evaluate_predictions(y_test.values, y_pred, y_prob)
    m['Model'] = name
    test_metrics.append(m)

test_df = pd.DataFrame(test_metrics).sort_values(by='ROC-AUC', ascending=False)
test_df

In [ ]:
plots = plot_evaluation_suite(trainer.models, X_test_proc, y_test.values, best_name, feature_names, output_dir='../reports')
from IPython.display import Image, display
for title, path in plots.items():
    print(f'=== {title.upper()} ===')
    display(Image(filename=path))